# Business Blind Spot Detector — pipeline walkthrough
Runs every stage on the **synthetic NovaCart demo data** and inspects the intermediate results. Run from the `notebooks/` folder with the project's virtualenv.

In [ ]:
import sys, pandas as pd
sys.path.insert(0, '..')
pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 80)
from ml.ingestion import standardize
customer, rep_c = standardize(pd.read_csv('../data/sample_customer_data.csv'), 'customer')
company, rep_k = standardize(pd.read_csv('../data/sample_company_data.csv'), 'company')
print(rep_c.rows_valid, 'customer records |', rep_k.rows_valid, 'company records')
customer.head()

## 1 · Preprocessing
Original text is preserved; negations, emojis and sentiment-bearing punctuation are kept.

In [ ]:
from ml.preprocessing import preprocess
clean, report = preprocess(customer)
print(report.to_dict())
clean[['text_original', 'text_clean', 'language']].head()

## 2 · Sentiment (VADER baseline + aspect level)
A review is split into clauses; each clause is scored and linked to a topic.

In [ ]:
from ml.sentiment import analyze_documents, vader_score
doc = analyze_documents(clean['text_clean'].tolist(), use_transformer=False)
print(doc['label'].value_counts(normalize=True).round(2).to_dict())
mixed = 'The phone camera is amazing, but the battery backup is disappointing.'
print('document-level:', vader_score(mixed))
from ml.text_utils import split_clauses
[(c, vader_score(c)) for c in split_clauses(mixed)]

## 3 · Run the complete pipeline
The same function the API and dashboard call.

In [ ]:
from backend.services.pipeline import run_pipeline
result = run_pipeline(customer, company, progress=lambda i, n, m: print(f'[{i}/{n}] {m}'), use_llm=False)
result.overview

## 4 · Topics discovered from the data (nothing hard-coded)

In [ ]:
result.topics[['label', 'mentions', 'unique_customers', 'avg_sentiment', 'negative_pct', 'importance', 'volume_change_pct']].round(2)

## 5 · What customers want — observed vs inferred

In [ ]:
result.expectations[['topic_label', 'kind', 'is_inferred', 'support_count', 'summary']].head(12)

## 6 · What the company communicates (analysed separately)

In [ ]:
result.company_themes[['label', 'n_messages', 'share_pct']]

## 7 · Semantic gap, scoring and classification
Blind Spot Score = Importance × Expectation Gap × Trend ÷ 10,000.

In [ ]:
cols = ['rank', 'topic_label', 'importance', 'coverage_norm', 'similarity_norm', 'gap', 'trend_used', 'score', 'severity', 'classification']
result.blind_spots[cols].round(1)

In [ ]:
import plotly.graph_objects as go
bs = result.blind_spots.sort_values('importance')
fig = go.Figure([go.Bar(y=bs.topic_label, x=bs.importance, name='Customer importance', orientation='h'),
                 go.Bar(y=bs.topic_label, x=bs.coverage_norm, name='Company coverage', orientation='h')])
fig.update_layout(barmode='group', title='Customer importance vs company coverage', height=400)
fig

## 8 · Re-scoring with different weights (no NLP re-run)

In [ ]:
from config.settings import ScoringConfig
from ml.scoring.rescore import rescore_blind_spots
raw = result.blind_spots.drop(columns=['rank', 'classification_label'])
alt = rescore_blind_spots(raw, ScoringConfig(w_coverage=1.0, w_similarity=0.0))
alt[['topic_label', 'score', 'severity']].round(1)

## 9 · Trends and recommendations

In [ ]:
print(result.meta['trend_available'], result.meta['trend_freq'])
display(result.trends.topic_summary[['topic_label', 'status', 'trend_score', 'volume_change_pct', 'volume_p_value']].round(2))
result.recommendations[['priority', 'category', 'topic_label', 'suggested_action']].head(8)